# 04. GO enrichment

In [14]:
from pathlib import Path
import sys

cwd = Path.cwd().resolve()
PROJECT_ROOT = cwd.parent if cwd.name == "notebooks" else cwd.parents[1] if cwd.name == "revision" else cwd
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from config import *
from reproducibility import seed_everything

seed_everything()

print("PROJECT_ROOT =", PROJECT_ROOT)
print("GLOBAL_SEED =", GLOBAL_SEED)


PROJECT_ROOT = /Users/jihopark/Desktop/MCDA_revision_final
GLOBAL_SEED = 20260829


This stage should be finalized only after the revised differential-expression results are confirmed. It is intentionally left as a controlled downstream step rather than reusing the original manuscript outputs.

In [15]:
import pandas as pd
import numpy as np
import re
from scipy.stats import fisher_exact
from statsmodels.stats.multitest import multipletests

DEG_FILE = RESULTS_DIR / "DEG_exploratory_candidates.csv"
GO_OUTPUT = RESULTS_DIR / "GO_enrichment_full_results.csv"
GO_MAPPING_OUTPUT = RESULTS_DIR / "GO_annotation_from_data2.csv"

print("DEG input:", DEG_FILE)
print("mRNA annotation:", MRNA_DATA2)

for p in [DEG_FILE, MRNA_DATA2]:
    print("FOUND" if p.exists() else "MISSING", "->", p)

if not DEG_FILE.exists():
    raise FileNotFoundError(DEG_FILE)

if not MRNA_DATA2.exists():
    raise FileNotFoundError(MRNA_DATA2)

DEG input: /Users/jihopark/Desktop/MCDA_revision_final/results/DEG_exploratory_candidates.csv
mRNA annotation: /Users/jihopark/Desktop/MCDA_revision_final/data/processed/mRNA/data2.xlsx
FOUND -> /Users/jihopark/Desktop/MCDA_revision_final/results/DEG_exploratory_candidates.csv
FOUND -> /Users/jihopark/Desktop/MCDA_revision_final/data/processed/mRNA/data2.xlsx


In [16]:
def split_go(value):
    if pd.isna(value):
        return []

    value = str(value).strip()

    if not value:
        return []

    return [
        x.strip()
        for x in re.split(
            r"\s*(?:///|//|;|\|)\s*",
            value
        )
        if x.strip()
    ]


data2 = pd.read_excel(MRNA_DATA2)

go_specs = [
    (
        "BP",
        "GO Biological Process ID",
        "GO Biological Process Term",
    ),
    (
        "CC",
        "GO Cellular Component ID",
        "GO Cellular Component Term",
    ),
    (
        "MF",
        "GO Molecular Function ID",
        "GO Molecular Function Term",
    ),
]

rows = []

for _, row in data2.iterrows():

    gene = row.get("Gene_Symbol")

    if pd.isna(gene):
        continue

    gene = str(gene).strip()

    if not gene:
        continue

    for ontology, id_col, term_col in go_specs:

        go_ids = split_go(row.get(id_col))
        go_terms = split_go(row.get(term_col))

        if not go_ids:
            continue

        # Keep ID even if the term field is incomplete.
        for i, go_id in enumerate(go_ids):

            go_term = (
                go_terms[i]
                if i < len(go_terms)
                else np.nan
            )

            rows.append({
                "Gene": gene,
                "GO_ID": go_id,
                "GO_term": go_term,
                "Ontology": ontology,
            })


go_ann = pd.DataFrame(rows)

go_ann = (
    go_ann
    .dropna(subset=["Gene", "GO_ID"])
    .drop_duplicates(
        ["Gene", "GO_ID", "Ontology"]
    )
    .reset_index(drop=True)
)

print("GO annotation rows:", len(go_ann))
print(
    "Unique GO-annotated genes:",
    go_ann["Gene"].nunique()
)

display(go_ann.head())

GO annotation rows: 319647
Unique GO-annotated genes: 18514


,Gene,GO_ID,GO_term,Ontology
0,OR4F5,GO:0007186,G-protein coupled receptor signaling pathway,BP
1,OR4F5,GO:0050907,detection of chemical stimulus involved in sen...,BP
2,OR4F5,GO:0050911,detection of chemical stimulus involved in sen...,BP
3,OR4F5,GO:0007165,signal transduction,BP
4,OR4F5,GO:0007608,sensory perception of smell,BP


In [17]:
deg = pd.read_csv(DEG_FILE)

candidate_genes = set(
    deg["Gene"]
    .dropna()
    .astype(str)
    .str.strip()
)

universe = set(
    go_ann["Gene"]
    .dropna()
    .astype(str)
)

candidates = candidate_genes & universe

print("Exploratory DEG candidates:", len(candidate_genes))
print("GO background genes:", len(universe))
print(
    "Candidates with GO annotation:",
    len(candidates)
)

Exploratory DEG candidates: 114
GO background genes: 18514
Candidates with GO annotation: 57


In [18]:
results = []

N = len(universe)
K = len(candidates)

for (go_id, ontology), group in go_ann.groupby(
    ["GO_ID", "Ontology"]
):

    term_genes = set(group["Gene"]) & universe

    hits = candidates & term_genes

    a = len(hits)

    if a == 0:
        continue

    b = K - a
    c = len(term_genes) - a
    d = N - a - b - c

    if min(a, b, c, d) < 0:
        continue

    odds_ratio, p_value = fisher_exact(
        [[a, b], [c, d]],
        alternative="greater"
    )

    terms = (
        group["GO_term"]
        .dropna()
        .astype(str)
        .unique()
    )

    go_term = (
        terms[0]
        if len(terms)
        else np.nan
    )

    results.append({
        "Ontology": ontology,
        "GO_ID": go_id,
        "GO_term": go_term,
        "candidate_hits": a,
        "candidate_total": K,
        "term_size": len(term_genes),
        "background_size": N,
        "odds_ratio": odds_ratio,
        "P.Value": p_value,
        "Genes": ";".join(sorted(hits)),
    })


go = pd.DataFrame(results)

print("GO terms with ≥1 candidate hit:", len(go))

GO terms with ≥1 candidate hit: 462


In [19]:
if go.empty:
    raise RuntimeError(
        "No GO enrichment results were generated."
    )

go["adj.P.Val"] = multipletests(
    go["P.Value"],
    method="fdr_bh"
)[1]

go["FDR_significant"] = (
    go["adj.P.Val"] < 0.05
)

go["single_gene_supported"] = (
    go["candidate_hits"] == 1
)

go = (
    go
    .sort_values(
        ["adj.P.Val", "P.Value"]
    )
    .reset_index(drop=True)
)

print("GO terms tested:", len(go))
print(
    "BH-FDR < 0.05:",
    int(go["FDR_significant"].sum())
)

print(
    "Single-gene-supported terms:",
    int(go["single_gene_supported"].sum())
)

display(
    go[
        [
            "Ontology",
            "GO_ID",
            "GO_term",
            "candidate_hits",
            "term_size",
            "odds_ratio",
            "P.Value",
            "adj.P.Val",
            "Genes",
        ]
    ].head(30)
)

GO terms tested: 462
BH-FDR < 0.05: 0
Single-gene-supported terms: 318


,Ontology,GO_ID,GO_term,candidate_hits,term_size,odds_ratio,P.Value,adj.P.Val,Genes
0,BP,GO:0006335,DNA replication-dependent nucleosome assembly,3,31,36.565476,0.000117,0.050183,CHAF1B;HIST1H4B;HIST1H4D
1,CC,GO:0000786,nucleosome,4,96,15.065628,0.000217,0.050183,HIST1H2AJ;HIST1H2BL;HIST1H4B;HIST1H4D
2,BP,GO:0035574,histone H4-K20 demethylation,2,15,51.591608,0.000953,0.095996,HIST1H4B;HIST1H4D
3,MF,GO:0035575,histone demethylase activity (H4-K20 specific),2,15,51.591608,0.000953,0.095996,HIST1H4B;HIST1H4D
4,BP,GO:0045653,negative regulation of megakaryocyte different...,2,17,44.707879,0.001229,0.095996,HIST1H4B;HIST1H4D
5,BP,GO:0006336,DNA replication-independent nucleosome assembly,2,26,27.928788,0.002886,0.095996,HIST1H4B;HIST1H4D
6,BP,GO:0002731,negative regulation of dendritic cell cytokine...,1,1,inf,0.003079,0.095996,JAK3
7,BP,GO:0045221,negative regulation of FasL biosynthetic process,1,1,inf,0.003079,0.095996,JAK3
8,BP,GO:0060562,epithelial tube morphogenesis,1,1,inf,0.003079,0.095996,PRKX
9,BP,GO:2000696,regulation of epithelial cell differentiation ...,1,1,inf,0.003079,0.095996,PRKX


In [20]:
go_ann.to_csv(
    GO_MAPPING_OUTPUT,
    index=False
)

go.to_csv(
    GO_OUTPUT,
    index=False
)

print("Saved:")
print(GO_MAPPING_OUTPUT)
print(GO_OUTPUT)

Saved:
/Users/jihopark/Desktop/MCDA_revision_final/results/GO_annotation_from_data2.csv
/Users/jihopark/Desktop/MCDA_revision_final/results/GO_enrichment_full_results.csv
